# 07 — Concentration Timing Robustness

Methodology gate: the primary specification uses concentration observed at formation month-end t. This notebook tests whether the conclusion changes when concentration is lagged to t−1, eliminating any concern about using a market-state measure determined at the same month-end as portfolio formation.

Primary specification remains unchanged. This is a timing robustness check only.


In [ ]:
from pathlib import Path
import sys

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.append(str(ROOT))

from src.clean_monthly import CleaningConfig, build_clean_monthly_panel, build_monthly_history_panel, build_market_state_panel
from src.concentration import calculate_monthly_concentration
from src.signals import attach_momentum_to_formation_universe, assign_signal_quintiles
from src.portfolio import quintile_returns, quintile_return_panel, monthly_rank_ic
from src.conditioning import merge_performance_and_concentration, primary_top10_spec, hhi_spec, lag_concentration_one_month


## 1. Rebuild validated inputs


In [ ]:
RAW = ROOT / 'data/raw/monthly_stock_common_equity_2000_2025_prod_v1.csv.gz'

clean, _ = build_clean_monthly_panel(RAW, CleaningConfig())
history = build_monthly_history_panel(RAW)
market_state = build_market_state_panel(RAW)
conc = calculate_monthly_concentration(market_state)

mom = attach_momentum_to_formation_universe(clean, history)
mom = assign_signal_quintiles(mom)
qret = quintile_returns(mom)
vw = quintile_return_panel(qret, return_col='VW_Return')
ew = quintile_return_panel(qret, return_col='EW_Return')
rank_ic = monthly_rank_ic(mom)


## 2. Build lagged concentration series


In [ ]:
conc_lag = lag_concentration_one_month(conc)
conc_lag.tail()


## 3. Primary VW spread with Top10Share at t−1


In [ ]:
vw_lag = merge_performance_and_concentration(vw, conc_lag)
lag_hac6, _ = primary_top10_spec(vw_lag, top10_col='Top10Share_Lag1', maxlags=6)
lag_hac6


## 4. HAC12 robustness


In [ ]:
lag_hac12, _ = primary_top10_spec(vw_lag, top10_col='Top10Share_Lag1', maxlags=12)
lag_hac12


## 5. HHI lagged robustness


In [ ]:
lag_hhi6, _ = hhi_spec(vw_lag, hhi_col='HHI_Lag1', maxlags=6)
lag_hhi6


## 6. Secondary outcomes with lagged Top10Share


In [ ]:
ew_lag = merge_performance_and_concentration(ew, conc_lag)
ew_lag_hac6, _ = primary_top10_spec(ew_lag, top10_col='Top10Share_Lag1', maxlags=6)
ew_lag_hac6


In [ ]:
ic_lag = merge_performance_and_concentration(rank_ic[['MthCalDt','RankIC']], conc_lag)
ic_lag_hac6, _ = primary_top10_spec(ic_lag, outcome_col='RankIC', top10_col='Top10Share_Lag1', maxlags=6)
ic_lag_hac6


## Interpretation rule

Compare sign, economic magnitude, confidence interval, and robustness with the contemporaneous-t primary specification. Do not replace the primary timing rule unless a genuine implementation problem is identified.
